In [ ]:
# ============================================================
# RandomizedSearchCV para Bagging(KNN) con TEST final
# + Guardado automático en carpeta fija "bagging_results"
# + ZIP dentro de esa carpeta (sin fechas)
# ============================================================

import time
import numpy as np
import pandas as pd

from pathlib import Path
from joblib import dump

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    f1_score, accuracy_score, confusion_matrix, classification_report
)
from scipy.stats import randint, uniform

# ==========================
# Cargar y separar X / y
# ==========================
Train = pd.read_csv("T_train_final_objetivo.csv")
Test  = pd.read_csv("T_test_final_objetivo.csv")

X_train = Train.iloc[:, :-1].to_numpy()
y_train = Train.iloc[:, -1].to_numpy().ravel()
X_test  = Test.iloc[:, :-1].to_numpy()
y_test  = Test.iloc[:, -1].to_numpy().ravel()

In [ ]:
# ---------- Ensamble base ----------
# n_jobs=1 para evitar paralelismo anidado (outer CV usa n_jobs=-1)
bag = BaggingClassifier(
    estimator=KNeighborsClassifier(),
    n_estimators=100,
    bootstrap=True,
    n_jobs=1,
    random_state=123
)

# ---------- Espacio aleatorio ----------
odd_ks = [k for k in range(3, 32, 2)]  # 3,5,...,31

param_distributions = {
    # --- KNN (modelo base) ---
    "estimator__n_neighbors": odd_ks,
    "estimator__weights": ["uniform", "distance"],
    "estimator__p": [1, 2],                      # 1=Manhattan, 2=Euclidiana
    # (Opcional velocidad/precisión):
    # "estimator__algorithm": ["auto", "kd_tree", "ball_tree", "brute"],
    # "estimator__leaf_size": randint(20, 51),

    # --- Bagging (ensamble) ---
    "n_estimators": randint(60, 181),            # 60..180
    "bootstrap": [True, False],
    "max_samples": uniform(0.6, 0.4),            # 0.6..1.0
    "max_features": uniform(0.6, 0.4),           # 0.6..1.0
    "bootstrap_features": [False, True],
}

In [ ]:
# ---------- Validación cruzada (reducida) ----------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

# ---------- Búsqueda aleatoria ----------
search = RandomizedSearchCV(
    estimator=bag,
    param_distributions=param_distributions,
    n_iter=40,                 # explora 40 combinaciones
    scoring="f1_micro",
    cv=cv,
    n_jobs=-1,                 # paralelismo a nivel de combinaciones/folds
    random_state=123,
    verbose=1,
    refit=True,
    return_train_score=False,
    error_score="raise"
)

t0 = time.perf_counter()
search.fit(X_train, y_train)
t1 = time.perf_counter()

print("\n=== MEJOR CONFIGURACIÓN (RandomizedSearch, 5-CV) ===")
print(search.best_params_)
print(f"Mejor F1_micro (CV): {search.best_score_:.4f}")
print(f"Tiempo total búsqueda: {t1 - t0:.2f} s")

best_model = search.best_estimator_
print("\nEstructura del mejor modelo:")
print(best_model)

In [ ]:
# ---------- Evaluación en TEST ----------
y_pred_test = best_model.predict(X_test)
acc_test = accuracy_score(y_test, y_pred_test)
f1_micro_test = f1_score(y_test, y_pred_test, average="micro")
cm = confusion_matrix(y_test, y_pred_test)
report_text = classification_report(y_test, y_pred_test, digits=4)

print(f"\n=== EVALUACIÓN EN TEST ===")
print(f"Accuracy (TEST): {acc_test:.4f}")
print(f"F1_micro (TEST): {f1_micro_test:.4f}")
print("Matriz de confusión (TEST):")
print(cm)
print("\nReporte de clasificación (TEST):")
print(report_text)

In [ ]:
# ============================================================
# Guardar TODO automáticamente en carpeta fija "bagging_results"
# y crear ZIP dentro de esa carpeta (sin fechas)
# ============================================================
import os, json, zipfile

# --- helpers para JSON (convertir tipos numpy a tipos Python) ---
def _to_py(obj):
    if isinstance(obj, (np.floating,)):   return float(obj)
    if isinstance(obj, (np.integer,)):    return int(obj)
    if isinstance(obj, (np.bool_,)):      return bool(obj)
    if isinstance(obj, (np.ndarray,)):    return obj.tolist()
    return obj

def _convert(d):
    if isinstance(d, dict):               return {k: _convert(v) for k, v in d.items()}
    if isinstance(d, (list, tuple)):      return [_convert(v) for v in d]
    return _to_py(d)

# Carpeta fija
OUTDIR = Path("bagging_results")
OUTDIR.mkdir(parents=True, exist_ok=True)

# ===== Modelo y parámetros =====
dump(best_model, OUTDIR / "bagging_knn_best_model.joblib")

best_params_json = _convert(search.best_params_)
with open(OUTDIR / "bagging_knn_best_params.json", "w", encoding="utf-8") as f:
    json.dump(best_params_json, f, ensure_ascii=False, indent=2)

# ===== cv_results completo =====
pd.DataFrame(search.cv_results_).to_csv(OUTDIR / "cv_results.csv", index=False)

# ===== Guardar evaluación en TEST =====
pd.DataFrame({"y_true": y_test, "y_pred": y_pred_test}).to_csv(
    OUTDIR / "y_test_and_pred.csv", index=False
)

cm_df = pd.DataFrame(cm)
cm_df.to_csv(OUTDIR / "confusion_matrix_test.csv", index=False)

with open(OUTDIR / "classification_report_test.txt", "w", encoding="utf-8") as f:
    f.write(report_text)

report_dict = classification_report(y_test, y_pred_test, output_dict=True)
with open(OUTDIR / "classification_report_test.json", "w", encoding="utf-8") as f:
    json.dump(_convert(report_dict), f, ensure_ascii=False, indent=2)

# ===== Resumen legible =====
summary_lines = []
summary_lines.append(f"Fitting {cv.get_n_splits()} folds for each of {search.n_iter} candidates, "
                     f"totalling {cv.get_n_splits() * search.n_iter} fits")
summary_lines.append("")
summary_lines.append("=== MEJOR CONFIGURACIÓN (RandomizedSearch, CV) ===")
summary_lines.append(json.dumps(best_params_json, ensure_ascii=False))
summary_lines.append(f"Mejor F1_micro (CV): {search.best_score_:.4f}")
summary_lines.append(f"Tiempo total búsqueda: {t1 - t0:.2f} s")
summary_lines.append("")
summary_lines.append("Estructura del mejor modelo:")
summary_lines.append(str(best_model))
summary_lines.append("")
summary_lines.append("=== EVALUACIÓN EN TEST ===")
summary_lines.append(f"Accuracy (TEST): {acc_test:.4f}")
summary_lines.append(f"F1_micro (TEST): {f1_micro_test:.4f}")
summary_lines.append("Matriz de confusión (TEST):")
summary_lines.append(cm_df.to_string(index=False))
summary_lines.append("")
summary_lines.append("Reporte de clasificación (TEST):")
summary_lines.append(report_text)

with open(OUTDIR / "summary.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

# ===== ZIP dentro de la carpeta =====
zip_path = OUTDIR / "bagging_results_bundle.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in OUTDIR.iterdir():
        if p.name == zip_path.name:
            continue  # no incluirse a sí mismo
        zf.write(p, arcname=p.name)

print("\n=== Artefactos guardados en:", OUTDIR.resolve())
for p in OUTDIR.iterdir():
    print(" -", p.name)
print(f"\nZIP generado (dentro de la carpeta): {zip_path.resolve()}")
